In [1]:
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

In [2]:
with open('Anna.txt', 'r') as file:
    text = file.read()

In [3]:
chars = tuple(set(text))
int_char = dict(enumerate(chars))
char_int = {ch : i for i, ch in int_char.items()}

encoded = np.array([char_int[x] for x in text])
encoded[:15]

array([60, 81, 52, 43, 62, 17, 13, 72, 20, 39, 39, 39, 42, 52, 43])

In [4]:
def one_hot_encode(arr, label):
    one_hot = np.array((arr.size, label))
    one_hot[arr.shape[0], arr.flatten()] = 1
    one_hot = one_hot.reshape((*arr.shape,label))
    
    return one_hot

In [5]:
def get_batches(arr, batch_size, seq_len):
    total_seq = batch_size*seq_len
    n_batch = len(arr)//(total_seq)
    arr = arr[:n_batch*(total_seq)]
    arr = arr.reshape((batch_size,-1))

    for n in range(0, arr.shape[1], seq_len):
        x = arr[:, n:n+seq_len]
        y = np.zeros_like(x)
        try:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, n+seq_len]
        except IndexError:
            y[:, :-1], y[:, -1] = x[:, 1:], arr[:, 0]
        yield x, y

In [6]:
batches = get_batches(encoded, 8, 50)
x, y = next(batches)

In [7]:
print('x\n', x[:10, :10])
print('y\n', y[:10, :10])

x
 [[60 81 52 43 62 17 13 72 20 39]
 [49 59 55 72 62 81 52 62 72 52]
 [17 55 30 72 59 13 72 52 72 82]
 [49 72 62 81 17 72 18 81  6 17]
 [72 49 52  7 72 81 17 13 72 62]
 [18 27 49 49  6 59 55 72 52 55]
 [72 16 55 55 52 72 81 52 30 72]
 [ 2  3 71 59 55 49 44 33 47 72]]
y
 [[81 52 43 62 17 13 72 20 39 39]
 [59 55 72 62 81 52 62 72 52 62]
 [55 30 72 59 13 72 52 72 82 59]
 [72 62 81 17 72 18 81  6 17 82]
 [49 52  7 72 81 17 13 72 62 17]
 [27 49 49  6 59 55 72 52 55 30]
 [16 55 55 52 72 81 52 30 72 49]
 [ 3 71 59 55 49 44 33 47 72 66]]


In [ ]:
class CharNN(nn.Module):    
    def __init__(self, tokens, n_layer=2, n_hidden=256, drop_prob=0.5, lr=0.001):
        super().__init__()

        self.n_layer = n_layer
        self.n_hidden = n_hidden
        self.lr = lr

        self.chars = tokens
        self.int_char = dict(enumerate(self.chars))
        self.char_int = {ch : i for i, ch in self.int_char.items()}

        self.lstm = nn.LSTM(len(self.chars), n_hidden, n_layer, dropout=drop_prob, batch_first=True)

        self.dropout = nn.Dropout(drop_prob)

        self.fc = nn.Linear(n_hidden, len(self.chars))

    def forward(self, x, hidden):
        r_output, hidden = self.lstm(x, hidden)

        out = self.dropout(r_output)
       
        out = out.contagious().view(-1, self.n_hidden)

        out = self.fc(out)

        return out, hidden
    
    def init_hidden (self, batch_size):
        
        weight = next(self.parameters()).data
        
        hidden = (weight.new(self.n_layer, batch_size, self.n_hidden).zero_(),
                  weight.new(self.n_layer, batch_size, self.n_hidden).zero_())
        
        return hidden

In [ ]:
def train(net, data, epoch=10, batch_size=10, seq_len=50, lr=0.01, clip=5, val_frac=0.1):

    net.train()

    opt = torch.optim.Adam(net.parameters(), lr=lr)
    crietrion = nn.CrossEntropyLoss()

    val_idx = int(len(data)*(1-val_frac))
    data, val_data = data[:val_idx], data[val_idx:]

    n_chars = len(net.chars)

    for n in range(epoch):
        h = net.init_hidden(batch_size)

        for x, y in get_batches(data, batch_size, seq_len):
            x = one_hot_encode(x, n_chars)
            input, target = torch.from_numpy(x), torch.from_numpy(y)

            h = tuple([each.data for each in h])

            net.zero_grad()
            output, h = net(input, h)
            loss = crietrion(output, target.view(batch_size*seq_len).long())

            loss.backward()
            nn.utils.clip_grad_norm_(net.parameters(), clip)
            opt.step()
        
        else:
            val_losses = []
            net.eval()
            val_h = net.init_hidden(batch_size)

            for x, y in get_batches(val_data, batch_size, seq_len):
                x = one_hot_encode(x,n_chars)
                inputs, targets = torch.from_numpy(x), torch.from_numpy(y)

                val_h = tuple([each.data for each in val_h])

                output, val_h = net(inputs, val_h)
                val_loss = crietrion(output, targets.view(batch_size*seq_len).long())

                val_losses.append(val_loss.items())

            net.train()

            print(f"Epoch: {n+1}, Train loss: {loss.items():4f}, Val loss: {np.mean(val_losses):4f}")